# exp065_typewell_supertype_cluster_cv_audit train

Common typewell discovery from train typewell CSV files. This notebook does not train a model or compute CV.

## Contents

1. Setup and configuration
2. Run exact / shifted NCC / DTW discovery
3. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os

from settings import EXPERIMENT_NAME, ExperimentPaths, load_config
from typewell_supertype_discovery import run_discovery

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None
SKIP_DTW = os.environ.get("EXPERIMENT_SKIP_DTW", "0") == "1"

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Purpose:", config["experiment"]["description"])
print("Route:", config["experiment"]["route"])
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Features:", paths.features_dir)
print("Max wells:", MAX_WELLS, "Skip DTW:", SKIP_DTW)


## 2. Run exact / shifted NCC / DTW discovery


In [ ]:
metrics = run_discovery(max_wells=MAX_WELLS, skip_dtw=SKIP_DTW)
print(json.dumps({
    "wells": metrics["wells"],
    "valid_signatures": metrics["valid_signatures"],
    "exact_unique_groups": metrics["exact_unique_groups"],
    "exact_duplicate_wells": metrics["exact_duplicate_wells"],
    "ncc_pair_rows": metrics["ncc_pair_rows"],
    "dtw_pair_rows": metrics["dtw_pair_rows"],
    "native_overlap_pair_rows": metrics["native_overlap_pair_rows"],
    "native_exact_containment_pair_rows": metrics["native_exact_containment_pair_rows"],
}, indent=2))


## 3. Metrics and artifacts


In [ ]:
for name, path in metrics["artifacts"].items():
    print(f"{name}: {path}")

print("Nearest thresholds to target group-count reference:")
print(json.dumps(metrics["nearest_thresholds_to_target_group_count"], indent=2))
print("Metrics written:", paths.metrics_path)
